# Load PPG/ABP Data and Extract Blood Pressure Labels

This notebook loads the PPG and ABP datasets produced by data/load_files.py from data/processed_dataset.npz. Each row is one complete 625-sample / 5-second window. The loader drops leftover samples that do not fill a complete window.

**Steps:**
1. Load PPG and ABP arrays from data/processed_dataset.npz.
2. For each ABP window, use the maximum value as systolic pressure (SBP) and the minimum value as diastolic pressure (DBP).
3. Join those two labels onto the matching PPG window by row index.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/processed_dataset.npz")

In [3]:
with np.load(DATA_PATH) as data:
    ppg_dataset = data["ppg"]
    abp_dataset = data["abp"]

ppg_dataset = pd.DataFrame(ppg_dataset)
abp_dataset = pd.DataFrame(abp_dataset)

print(f"ppg_dataset shape: {ppg_dataset.shape}")
print(f"abp_dataset shape: {abp_dataset.shape}")

ppg_dataset shape: (528828, 625)
abp_dataset shape: (528828, 625)


## Extract systolic and diastolic values from each ABP window

For each 625-sample ABP window: systolic pressure (SBP) is the max value in the window, and diastolic pressure (DBP) is the min value in the window, computed directly on the whole abp_dataset array with vectorized pandas operations.

In [4]:
abp_dataset["systolic pressure"] = abp_dataset.max(axis=1)
abp_dataset["diastolic pressure"] = abp_dataset.min(axis=1)

# join the 2 new columns onto the ppg dataset 
ppg_dataset["systolic pressure"] = abp_dataset["systolic pressure"]
ppg_dataset["diastolic pressure"] = abp_dataset["diastolic pressure"]
print(f"ppg_dataset shape: {ppg_dataset.shape}")

ppg_dataset shape: (528828, 627)


In [5]:
ppg_dataset

,0,1,2,3,4,5,6,7,8,9,...,617,618,619,620,621,622,623,624,systolic pressure,diastolic pressure
0,1.759531,1.718475,1.684262,1.657869,1.637341,1.615836,1.593353,1.570870,1.549365,1.526882,...,1.734115,1.698925,1.668622,1.641251,1.615836,1.591398,1.565005,1.539589,125.187439,65.597632
1,1.513196,1.485826,1.463343,1.443793,1.419355,1.393939,1.369501,1.345064,1.321603,1.300098,...,1.211144,1.188661,1.164223,1.141740,1.130010,1.141740,1.189638,1.283480,127.434268,67.307176
2,1.427175,1.584555,1.757576,1.968719,2.190616,2.399804,2.579668,2.725318,2.833822,2.910068,...,2.938416,2.957967,2.956989,2.940371,2.911046,2.868035,2.819159,2.763441,127.190048,68.381746
3,2.694037,2.611926,2.518084,2.415445,2.306940,2.196481,2.088954,1.986315,1.892473,1.807429,...,1.893451,1.815249,1.747801,1.689150,1.640274,1.605083,1.577713,1.552297,124.308245,66.037229
4,1.527859,1.504399,1.480938,1.457478,1.432063,1.405670,1.378299,1.350929,1.324536,1.304985,...,1.054741,1.038123,1.021505,1.002933,0.989247,0.975562,0.956012,0.938416,128.508838,67.551396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528823,1.165200,1.148583,1.136852,1.130010,1.127077,1.128055,1.130987,1.134897,1.137830,1.140762,...,2.327468,2.259042,2.186706,2.110459,2.040078,1.971652,1.895406,1.815249,101.986490,50.309427
528824,1.734115,1.654936,1.579668,1.511241,1.450635,1.398827,1.355816,1.323558,1.301075,1.288368,...,2.644184,2.584555,2.522972,2.448680,2.364614,2.282502,2.199413,2.114370,101.693425,50.065207
528825,2.863148,2.840665,2.810362,2.774194,2.731183,2.683284,2.631476,2.576735,2.521017,2.464321,...,2.229717,2.220919,2.217009,2.217986,2.220919,2.224829,2.226784,2.225806,112.732193,50.358272
528826,2.221896,2.217986,2.214076,2.211144,2.211144,2.214076,2.219941,2.229717,2.240469,2.253177,...,2.153470,2.071359,1.993157,1.919844,1.852395,1.790811,1.736070,1.687195,114.344048,50.993245


In [6]:
print(ppg_dataset.isnull().values.any())
print(abp_dataset.isnull().values.any())

ppg_neg_rows = {}
abp_neg_rows = {}

# ppg_neg_rows = {i:len([val for val in arr if val <= 0]) for i, arr in ppg_dataset.values if len([val for val in arr if val <= 0]) > 0} # dict is row : count of negative values
# abp_neg_rows = {i:len([val for val in arr if val <= 0]) for i, arr in abp_dataset.values if len([val for val in arr if val <= 0]) > 0}
# ppg_neg_rows = {}
for i, arr in enumerate(ppg_dataset.values):
    negs = {i:val for i, val in enumerate(arr) if val <= 0}
    if len(negs) > 0:
        ppg_neg_rows[i] = negs

for i, arr in enumerate(abp_dataset.values):
    negs = {i:val for i, val in enumerate(arr) if val <= 0}
    if len(negs) > 0:
        abp_neg_rows[i] = negs

# # ppg_neg_values = [{len([val for val in arr if val <= 0])} for arr in ppg_dataset.values if len([val for val in arr if val <= 0]) > 0]
# ppg_neg_values = [len(negs) for negs in ppg_neg_rows.values()]

False
False


In [7]:
print("PPG number of neg rows: ", len(ppg_neg_rows))
print("ABP number of neg rows: ", len(abp_neg_rows))

leniency_percent = 0.05 # 627 entries per row, if 5% entries are inadmissible (<=0, null), we will remove row from dataset
num_entries = 627
leniency = int(num_entries * leniency_percent)

PPG number of neg rows:  5407
ABP number of neg rows:  0


In [8]:
dropped_num = 0
to_drop_rows = []

for row in ppg_neg_rows:
    if len(ppg_neg_rows[row]) > leniency:
        ppg_dataset.drop(row, inplace=True)
        abp_dataset.drop(row, inplace=True)
        to_drop_rows.append(row)
        dropped_num += 1
        print("dropped row: ", row)

print("Total rows dropped: ", dropped_num)

dropped row:  843
dropped row:  9666
dropped row:  19683
dropped row:  33442
dropped row:  60680
dropped row:  66702
dropped row:  89609
dropped row:  96165
dropped row:  103501
dropped row:  108575
dropped row:  120406
dropped row:  132910
dropped row:  133380
dropped row:  142945
dropped row:  149185
dropped row:  151494
dropped row:  162865
dropped row:  216678
dropped row:  218476
dropped row:  241182
dropped row:  246554
dropped row:  257243
dropped row:  274181
dropped row:  279631
dropped row:  280696
dropped row:  280783
dropped row:  304199
dropped row:  305204
dropped row:  328617
dropped row:  333582
dropped row:  333776
dropped row:  337119
dropped row:  349752
dropped row:  381339
dropped row:  382049
dropped row:  383363
dropped row:  389363
dropped row:  396340
dropped row:  402072
dropped row:  402767
dropped row:  403384
dropped row:  404718
dropped row:  404733
dropped row:  406614
dropped row:  407812
dropped row:  409655
dropped row:  419763
dropped row:  466445
dro

In [9]:
for row in to_drop_rows:
    ppg_neg_rows.pop(row)

In [10]:
for row in ppg_neg_rows:
    mean = ppg_dataset.iloc[row].mean()
    ppg_dataset.iloc[row].fillna(value=mean, inplace=True)

/var/folders/8l/b0ljklrx1g59s9ht0jv85dfw0000gn/T/ipykernel_7972/3942558586.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ppg_dataset.iloc[row].fillna(value=mean, inplace=True)


In [11]:
print(ppg_dataset.shape)
ppg_dataset.head()

(528775, 627)


,0,1,2,3,4,5,6,7,8,9,...,617,618,619,620,621,622,623,624,systolic pressure,diastolic pressure
0,1.759531,1.718475,1.684262,1.657869,1.637341,1.615836,1.593353,1.570870,1.549365,1.526882,...,1.734115,1.698925,1.668622,1.641251,1.615836,1.591398,1.565005,1.539589,125.187439,65.597632
1,1.513196,1.485826,1.463343,1.443793,1.419355,1.393939,1.369501,1.345064,1.321603,1.300098,...,1.211144,1.188661,1.164223,1.141740,1.130010,1.141740,1.189638,1.283480,127.434268,67.307176
2,1.427175,1.584555,1.757576,1.968719,2.190616,2.399804,2.579668,2.725318,2.833822,2.910068,...,2.938416,2.957967,2.956989,2.940371,2.911046,2.868035,2.819159,2.763441,127.190048,68.381746
3,2.694037,2.611926,2.518084,2.415445,2.306940,2.196481,2.088954,1.986315,1.892473,1.807429,...,1.893451,1.815249,1.747801,1.689150,1.640274,1.605083,1.577713,1.552297,124.308245,66.037229
4,1.527859,1.504399,1.480938,1.457478,1.432063,1.405670,1.378299,1.350929,1.324536,1.304985,...,1.054741,1.038123,1.021505,1.002933,0.989247,0.975562,0.956012,0.938416,128.508838,67.551396


## Raw QC Flags Without Dropping Rows

This section reloads the original `processed_dataset.npz` into separate variables and adds raw quality-control flags. It does not modify the earlier `ppg_dataset` or `abp_dataset` objects, and it does not delete rows from the full saved table.

Use the full flagged table for auditing, then use the filtered table for model experiments if the thresholds are acceptable.

In [12]:
RAW_QC_OUTPUT_DIR = Path("../data/features/read_data_raw_qc")
RAW_QC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(DATA_PATH) as data:
    ppg_raw_windows = data["ppg"].astype(float, copy=False)
    abp_raw_windows = data["abp"].astype(float, copy=False)

print(f"Raw PPG windows: {ppg_raw_windows.shape}")
print(f"Raw ABP windows: {abp_raw_windows.shape}")

Raw PPG windows: (528828, 625)
Raw ABP windows: (528828, 625)


In [13]:
def longest_true_run(mask):
    """Return the longest consecutive True run in a 1D boolean array."""
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():
        return 0
    padded = np.concatenate(([False], mask, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return int((changes[1::2] - changes[::2]).max())


raw_qc = pd.DataFrame({
    "window_index": np.arange(ppg_raw_windows.shape[0]),
    "systolic_pressure_target": np.max(abp_raw_windows, axis=1),
    "diastolic_pressure_target": np.min(abp_raw_windows, axis=1),
    "abp_window_range_qc": np.ptp(abp_raw_windows, axis=1),
    "ppg_nonfinite_count_qc": (~np.isfinite(ppg_raw_windows)).sum(axis=1),
    "abp_nonfinite_count_qc": (~np.isfinite(abp_raw_windows)).sum(axis=1),
    "ppg_nonpositive_count_qc": (ppg_raw_windows <= 0).sum(axis=1),
    "ppg_zero_count_qc": (ppg_raw_windows == 0).sum(axis=1),
    "ppg_zero_fraction_qc": (ppg_raw_windows == 0).mean(axis=1),
    "ppg_raw_min_qc": np.min(ppg_raw_windows, axis=1),
    "ppg_raw_max_qc": np.max(ppg_raw_windows, axis=1),
    "ppg_raw_range_qc": np.ptp(ppg_raw_windows, axis=1),
    "ppg_raw_std_qc": np.std(ppg_raw_windows, axis=1),
})

raw_qc["ppg_longest_zero_run_qc"] = [
    longest_true_run(row == 0) for row in ppg_raw_windows
]

display(raw_qc.head())

,window_index,systolic_pressure_target,diastolic_pressure_target,abp_window_range_qc,ppg_nonfinite_count_qc,abp_nonfinite_count_qc,ppg_nonpositive_count_qc,ppg_zero_count_qc,ppg_zero_fraction_qc,ppg_raw_min_qc,ppg_raw_max_qc,ppg_raw_range_qc,ppg_raw_std_qc,ppg_longest_zero_run_qc
0,0,125.187439,65.597632,59.589807,0,0,0,0,0.0,1.041056,3.041056,2.000000,0.601276,0
1,1,127.434268,67.307176,60.127092,0,0,0,0,0.0,0.945259,2.990225,2.044966,0.613709,0
2,2,127.190048,68.381746,58.808302,0,0,0,0,0.0,0.985337,2.992180,2.006843,0.627820,0
3,3,124.308245,66.037229,58.271016,0,0,0,0,0.0,0.920821,3.022483,2.101662,0.615459,0
4,4,128.508838,67.551396,60.957442,0,0,0,0,0.0,0.938416,2.986315,2.047898,0.620946,0


### Raw Outlier Rules

The percentile thresholds below are dataset-relative. They mark review candidates, not automatic data deletion.

In [14]:
low_range_threshold = raw_qc["ppg_raw_range_qc"].quantile(0.01)
high_range_threshold = raw_qc["ppg_raw_range_qc"].quantile(0.99)
low_std_threshold = raw_qc["ppg_raw_std_qc"].quantile(0.01)
high_std_threshold = raw_qc["ppg_raw_std_qc"].quantile(0.99)

raw_qc["raw_nonfinite_outlier_qc"] = (raw_qc["ppg_nonfinite_count_qc"] > 0) | (raw_qc["abp_nonfinite_count_qc"] > 0)
raw_qc["raw_nonpositive_ppg_outlier_qc"] = raw_qc["ppg_nonpositive_count_qc"] > int(0.05 * ppg_raw_windows.shape[1])
raw_qc["raw_zero_ppg_outlier_qc"] = raw_qc["ppg_zero_count_qc"] > 0
raw_qc["raw_long_zero_run_outlier_qc"] = raw_qc["ppg_longest_zero_run_qc"] >= 32
raw_qc["raw_flat_ppg_outlier_qc"] = raw_qc["ppg_raw_range_qc"] == 0
raw_qc["raw_low_ppg_range_outlier_qc"] = raw_qc["ppg_raw_range_qc"] <= low_range_threshold
raw_qc["raw_high_ppg_range_outlier_qc"] = raw_qc["ppg_raw_range_qc"] >= high_range_threshold
raw_qc["raw_low_ppg_std_outlier_qc"] = raw_qc["ppg_raw_std_qc"] <= low_std_threshold
raw_qc["raw_high_ppg_std_outlier_qc"] = raw_qc["ppg_raw_std_qc"] >= high_std_threshold
raw_qc["raw_abp_label_outlier_qc"] = (
    raw_qc["systolic_pressure_target"].gt(250)
    | raw_qc["diastolic_pressure_target"].lt(40)
    | raw_qc["abp_window_range_qc"].lt(5)
)

raw_outlier_columns = [column for column in raw_qc.columns if column.startswith("raw_") and column.endswith("_outlier_qc")]
raw_qc["any_raw_outlier_qc"] = raw_qc[raw_outlier_columns].any(axis=1)
raw_qc["raw_outlier_reason_qc"] = raw_qc[raw_outlier_columns].apply(
    lambda row: ";".join(column.replace("raw_", "").replace("_outlier_qc", "") for column, flagged in row.items() if flagged),
    axis=1,
)

print("Raw QC thresholds:")
print(f"  PPG range <= {low_range_threshold:.6g} or >= {high_range_threshold:.6g}")
print(f"  PPG std   <= {low_std_threshold:.6g} or >= {high_std_threshold:.6g}")
print(f"  More than 5% nonpositive PPG samples: > {int(0.05 * ppg_raw_windows.shape[1])} samples")

raw_qc_summary = raw_qc[raw_outlier_columns + ["any_raw_outlier_qc"]].sum().sort_values(ascending=False).to_frame("window_count")
raw_qc_summary["window_percent"] = 100 * raw_qc_summary["window_count"] / len(raw_qc)
display(raw_qc_summary)

Raw QC thresholds:
  PPG range <= 0.484848 or >= 3.31574
  PPG std   <= 0.125341 or >= 0.750007
  More than 5% nonpositive PPG samples: > 31 samples


,window_count,window_percent
any_raw_outlier_qc,17277,3.267036
raw_low_ppg_range_outlier_qc,5425,1.025853
raw_zero_ppg_outlier_qc,5407,1.022450
raw_high_ppg_range_outlier_qc,5293,1.000893
raw_low_ppg_std_outlier_qc,5289,1.000136
raw_high_ppg_std_outlier_qc,5289,1.000136
raw_nonpositive_ppg_outlier_qc,53,0.010022
raw_nonfinite_outlier_qc,0,0.000000
raw_long_zero_run_outlier_qc,0,0.000000
raw_flat_ppg_outlier_qc,0,0.000000


In [15]:
full_raw_qc_path = RAW_QC_OUTPUT_DIR / "labeled_windows_with_raw_qc.csv"
model_ready_raw_qc_path = RAW_QC_OUTPUT_DIR / "model_ready_raw_qc_windows.csv"
raw_qc_summary_path = RAW_QC_OUTPUT_DIR / "raw_qc_summary.csv"

raw_qc.to_csv(full_raw_qc_path, index=False)
raw_qc.loc[~raw_qc["any_raw_outlier_qc"]].to_csv(model_ready_raw_qc_path, index=False)
raw_qc_summary.to_csv(raw_qc_summary_path)

print(f"Saved full labeled + raw QC table: {full_raw_qc_path}")
print(f"Saved filtered model-ready raw QC table: {model_ready_raw_qc_path}")
print(f"Saved raw QC summary: {raw_qc_summary_path}")
print(f"Model-ready windows after raw QC: {(~raw_qc['any_raw_outlier_qc']).sum():,} / {len(raw_qc):,}")

Saved full labeled + raw QC table: ../data/features/read_data_raw_qc/labeled_windows_with_raw_qc.csv
Saved filtered model-ready raw QC table: ../data/features/read_data_raw_qc/model_ready_raw_qc_windows.csv
Saved raw QC summary: ../data/features/read_data_raw_qc/raw_qc_summary.csv
Model-ready windows after raw QC: 511,551 / 528,828
